# 13장. 외부 데이터로 분석을 확장하기

이 노트북은 `book/chapters/ch13_external_data_collection.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 내부 데이터만으로 답하기 어려운 질문을 외부 데이터로 확장하되, **출처, 수집 시점, API Key 관리, 크롤링 정책, 내부 데이터와의 연결 기준**을 함께 점검하는 것입니다.

주의: 현재 노트북 파일명은 기존 목차 기준의 `ch13_make_automation.ipynb`이지만, 실제 13장 강의안 내용은 외부 데이터 수집입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 외부 데이터 저장 폴더는 `data/external/`입니다.
- 결과 요약과 체크리스트는 `reports/` 폴더에 저장합니다.
- API Key는 코드에 직접 입력하지 않고 `.env` 또는 환경변수로 관리합니다.
- 네이버 API 호출은 인증 정보가 있을 때만 실행합니다.
- 크롤링 예시는 구조 이해용이며, 실제 사이트 적용 전 이용약관과 robots.txt를 확인해야 합니다.


## 1. 외부 데이터가 필요한 이유

내부 온라인 쇼핑몰 데이터만 보면 매출, 주문, 고객, 상품 현황은 알 수 있습니다. 하지만 매출이 왜 증가했는지, 특정 지역에서 구매가 많은 이유가 무엇인지, 특정 키워드 관심도가 높아졌는지는 내부 데이터만으로 설명하기 어렵습니다.

| 내부 데이터 질문 | 함께 보면 좋은 외부 데이터 |
|---|---|
| 특정 월 매출이 왜 증가했을까? | 공휴일, 날씨, 이벤트, 검색 트렌드 |
| 특정 지역 고객 구매가 많은 이유는 무엇일까? | 지역 인구, 관광지, 상권 정보 |
| 여행 상품 추천 앱을 만들려면 어떤 데이터가 필요할까? | 관광지, 숙박, 음식점, 위치 데이터 |
| 특정 키워드의 관심도가 높아지고 있을까? | 검색 API, 뉴스, 블로그, SNS 데이터 |
| 공공 API를 활용한 서비스 기획이 가능할까? | 공공데이터포털, 한국관광공사 OpenAPI |

외부 데이터는 분석을 풍부하게 만들지만, 출처와 연결 기준이 불명확하면 오히려 해석을 흐릴 수 있습니다.


## 2. 패키지와 경로 설정

외부 데이터 수집에는 `requests`, `BeautifulSoup`, `python-dotenv`가 자주 사용됩니다. 패키지가 설치되어 있지 않다면 해당 기능은 설치 후 실행하세요.


In [ ]:
from pathlib import Path
import os
import json
from datetime import datetime
from urllib.parse import urlparse

import pandas as pd

try:
    import requests
except ImportError:
    requests = None

try:
    from bs4 import BeautifulSoup
except ImportError:
    BeautifulSoup = None

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'
REPORT_DIR = PROJECT_ROOT / 'reports'

EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('프로젝트 루트:', PROJECT_ROOT)
print('외부 데이터 폴더:', EXTERNAL_DIR)
print('보고서 폴더:', REPORT_DIR)


## 3. 외부 데이터 수집 방법 비교

외부 데이터를 가져오는 대표적인 방법은 파일 다운로드, API 호출, 크롤링입니다. 가장 안정적인 방법은 공식 API나 공식 다운로드 파일을 사용하는 것입니다.


In [ ]:
collection_method_summary = pd.DataFrame({
    'method': ['파일 다운로드', 'API 호출', '크롤링'],
    'description': [
        'CSV, Excel, JSON 파일을 직접 내려받아 사용',
        '정해진 주소와 파라미터로 데이터를 요청',
        '웹페이지 HTML에서 필요한 정보를 추출',
    ],
    'example': [
        '공공데이터포털 CSV, 통계청 Excel',
        '공공데이터 API, 네이버 검색 API',
        '공개 웹페이지의 표, 제목, 링크',
    ],
    'priority': [
        '안정적이며 우선 검토',
        '공식 문서와 인증 필요',
        '정책 확인 후 제한적으로 사용',
    ],
})

collection_method_summary.to_csv(REPORT_DIR / 'ch13_collection_method_summary.csv', index=False, encoding='utf-8-sig')
collection_method_summary


## 4. 외부 데이터 후보 계획표 만들기

외부 데이터 수집은 분석 질문에서 시작해야 합니다. 먼저 내부 데이터 질문과 연결 가능한 외부 데이터 후보를 정리합니다.


In [ ]:
external_data_plan = pd.DataFrame({
    'internal_question': [
        '특정 월 매출이 왜 증가했을까?',
        '특정 지역 고객 구매가 많은 이유는 무엇일까?',
        '여행 상품 추천 앱을 만들려면 어떤 데이터가 필요할까?',
        '특정 키워드 관심도가 높아지고 있을까?',
        '공공 API를 활용한 서비스 기획이 가능할까?',
    ],
    'external_data_candidate': [
        '공휴일, 날씨, 이벤트, 검색 트렌드',
        '지역 인구, 관광지, 상권 정보',
        '관광지, 숙박, 음식점, 위치 데이터',
        '검색 API, 뉴스, 블로그, SNS 데이터',
        '공공데이터포털, 한국관광공사 OpenAPI',
    ],
    'connection_key': [
        '날짜 또는 월',
        '지역명 또는 행정구역 코드',
        '위치, 지역 코드, 카테고리',
        '키워드, 날짜',
        'API 제공 키, 지역, 카테고리',
    ],
    'caution': [
        '매출 증가 원인을 단정하지 말 것',
        '지역명 표기 차이 확인',
        '상업적 이용 조건과 출처 확인',
        '검색 결과가 실제 수요를 대표한다고 단정하지 말 것',
        'API 문서와 요청 제한 확인',
    ],
})

external_data_plan.to_csv(REPORT_DIR / 'ch13_external_data_plan.csv', index=False, encoding='utf-8-sig')
external_data_plan


## 5. API Key 안전 관리

API Key는 코드에 직접 쓰지 않습니다. `.env` 파일에 저장하고, `.gitignore`에 포함해야 합니다. 이 노트북에서는 실제 Key 값을 출력하지 않고 로드 여부만 확인합니다.

`.env` 예시:

```text
PUBLIC_DATA_API_KEY=YOUR_PUBLIC_DATA_API_KEY
NAVER_CLIENT_ID=YOUR_NAVER_CLIENT_ID
NAVER_CLIENT_SECRET=YOUR_NAVER_CLIENT_SECRET
```


In [ ]:
if load_dotenv is not None:
    load_dotenv(PROJECT_ROOT / '.env')

env_status = pd.DataFrame([
    {'env_key': 'PUBLIC_DATA_API_KEY', 'loaded': os.getenv('PUBLIC_DATA_API_KEY') is not None},
    {'env_key': 'NAVER_CLIENT_ID', 'loaded': os.getenv('NAVER_CLIENT_ID') is not None},
    {'env_key': 'NAVER_CLIENT_SECRET', 'loaded': os.getenv('NAVER_CLIENT_SECRET') is not None},
])

env_status.to_csv(REPORT_DIR / 'ch13_env_key_status.csv', index=False, encoding='utf-8-sig')
env_status


## 6. 공공데이터 파일 읽기 예시

공공데이터 CSV 파일을 직접 내려받아 `data/external/`에 저장한 경우, pandas로 읽고 구조를 확인할 수 있습니다. 아래 코드는 파일이 있을 때만 실행됩니다.


In [ ]:
public_data_path = EXTERNAL_DIR / 'public_data_sample.csv'

if public_data_path.exists():
    public_df = pd.read_csv(public_data_path)
    print('public_df:', public_df.shape)
    display(public_df.head())
else:
    public_df = None
    print('public_data_sample.csv 파일이 아직 없습니다. 공공데이터 CSV를 내려받아 data/external/에 저장하면 이 셀을 실행할 수 있습니다.')


## 7. 네이버 블로그 검색 API 구조

네이버 블로그 검색 API는 키워드 기반 블로그 검색 결과를 가져올 수 있습니다. 인증 정보가 없으면 실제 호출은 건너뜁니다. 검색 결과는 전체 여론이나 실제 수요를 대표한다고 단정하면 안 됩니다.


In [ ]:
def clean_html_text(text):
    if pd.isna(text):
        return ''
    text = str(text)
    if BeautifulSoup is None:
        return text.replace('<b>', '').replace('</b>', '')
    return BeautifulSoup(text, 'html.parser').get_text()


def search_naver_blog(query, display=10, start=1, sort='sim'):
    if requests is None:
        raise ImportError('requests 패키지가 필요합니다.')

    naver_client_id = os.getenv('NAVER_CLIENT_ID')
    naver_client_secret = os.getenv('NAVER_CLIENT_SECRET')

    if not naver_client_id or not naver_client_secret:
        raise ValueError('네이버 API 인증 정보가 없습니다. .env 파일을 확인하세요.')

    url = 'https://openapi.naver.com/v1/search/blog.json'
    headers = {
        'X-Naver-Client-Id': naver_client_id,
        'X-Naver-Client-Secret': naver_client_secret,
    }
    params = {
        'query': query,
        'display': display,
        'start': start,
        'sort': sort,
    }

    response = requests.get(url, headers=headers, params=params, timeout=10)
    print('status_code:', response.status_code)
    response.raise_for_status()
    return response.json()


if os.getenv('NAVER_CLIENT_ID') and os.getenv('NAVER_CLIENT_SECRET'):
    result = search_naver_blog('제주 여행', display=10)
    naver_blog_df = pd.DataFrame(result.get('items', []))
    if not naver_blog_df.empty:
        naver_blog_df['title_clean'] = naver_blog_df['title'].apply(clean_html_text)
        naver_blog_df['description_clean'] = naver_blog_df['description'].apply(clean_html_text)
        naver_blog_df.to_csv(EXTERNAL_DIR / 'naver_blog_search_jeju.csv', index=False, encoding='utf-8-sig')
    display(naver_blog_df.head())
else:
    print('네이버 API 인증 정보가 없습니다. 실제 호출은 건너뜁니다.')


## 8. 기본 크롤링 구조 이해하기

크롤링은 공식 API나 다운로드 파일이 없을 때 공개 페이지에서 제한적으로 사용합니다. 실제 사이트를 대상으로 실행하기 전에는 반드시 이용약관, robots.txt, 저작권, 개인정보 여부를 확인해야 합니다.


In [ ]:
def fetch_page(url):
    if requests is None:
        raise ImportError('requests 패키지가 필요합니다.')

    parsed = urlparse(url)
    if parsed.scheme not in {'http', 'https'}:
        raise ValueError('http 또는 https URL만 요청할 수 있습니다.')

    headers = {
        'User-Agent': 'Mozilla/5.0 (compatible; DataAnalysisCourseBot/1.0; educational use)',
    }
    response = requests.get(url, headers=headers, timeout=10)
    print('status_code:', response.status_code)
    response.raise_for_status()
    return response.text


def extract_title_and_links(html, base_url=''):
    if BeautifulSoup is None:
        raise ImportError('beautifulsoup4 패키지가 필요합니다.')

    soup = BeautifulSoup(html, 'html.parser')
    page_title = soup.title.get_text(strip=True) if soup.title else ''
    links = []

    for a_tag in soup.find_all('a'):
        text = a_tag.get_text(strip=True)
        href = a_tag.get('href')
        if text or href:
            links.append({'page_title': page_title, 'text': text, 'href': href, 'base_url': base_url})

    return page_title, pd.DataFrame(links)


sample_url = 'https://example.com'

if requests is not None and BeautifulSoup is not None:
    html = fetch_page(sample_url)
    page_title, links_df = extract_title_and_links(html, base_url=sample_url)
    links_df.to_csv(EXTERNAL_DIR / 'scraped_example_links.csv', index=False, encoding='utf-8-sig')
    print('page_title:', page_title)
    display(links_df.head())
else:
    print('requests 또는 beautifulsoup4 패키지가 없어 크롤링 예시는 건너뜁니다.')


## 9. 외부 데이터를 내부 데이터와 연결하는 기준

외부 데이터는 내부 데이터와 연결 기준이 있어야 분석에 활용할 수 있습니다. 날짜, 지역, 카테고리, 키워드, 위치 중 어떤 기준으로 연결할지 먼저 정리합니다.


In [ ]:
external_integration_plan = pd.DataFrame({
    'connection_key': ['날짜', '지역', '상품 카테고리', '키워드', '위치'],
    'internal_data_example': [
        '월별 매출, 주문 일자',
        '고객 city',
        '상품 category',
        '상품명, 카테고리명',
        '고객 지역, 관광지 위치',
    ],
    'external_data_example': [
        '공휴일, 날씨, 검색 트렌드',
        '지역 인구, 관광지, 상권 정보',
        '검색 키워드, 뉴스 데이터',
        '블로그/뉴스 검색 결과',
        '위도·경도 기반 관광지/상권 데이터',
    ],
    'validation_check': [
        '날짜 형식과 분석 단위 일치 확인',
        '지역명 표기와 행정구역 단위 확인',
        '카테고리 매핑 기준 확인',
        '검색어 대표성 및 수집 시점 확인',
        '좌표계와 거리 기준 확인',
    ],
})

external_integration_plan.to_csv(REPORT_DIR / 'ch13_external_integration_plan.csv', index=False, encoding='utf-8-sig')
external_integration_plan


## 10. 외부 데이터 수집 체크리스트 만들기

외부 데이터를 수집할 때는 출처, 사용 조건, API Key, 요청 제한, 크롤링 정책, 내부 데이터 연결 기준을 함께 확인해야 합니다.


In [ ]:
external_data_checklist = pd.DataFrame({
    'check_item': [
        '분석 질문에 필요한 외부 데이터인가?',
        '공식 API 또는 다운로드 파일을 우선 확인했는가?',
        '데이터 출처와 제공 기관을 기록했는가?',
        '업데이트 주기와 수집 시점을 기록했는가?',
        'API Key를 .env에 저장했는가?',
        'API Key를 출력하거나 GitHub에 올리지 않았는가?',
        '요청 URL과 파라미터를 공식 문서 기준으로 확인했는가?',
        '응답 상태 코드와 오류 처리를 포함했는가?',
        '원본 응답 또는 원본 파일을 보관했는가?',
        'DataFrame 변환 후 컬럼과 행 수를 확인했는가?',
        '기존 데이터와 연결할 기준 컬럼이 있는가?',
        '날짜, 지역, 키워드 표기를 맞췄는가?',
        '크롤링 대상 사이트의 정책을 확인했는가?',
        '개인정보나 민감정보를 수집하지 않았는가?',
        'LLM이 만든 코드와 실제 공식 문서를 비교했는가?',
    ],
    'status': ['□'] * 15,
    'memo': [''] * 15,
})

external_data_checklist.to_csv(REPORT_DIR / 'ch13_external_data_checklist.csv', index=False, encoding='utf-8-sig')
external_data_checklist


## 11. 외부 데이터 수집 로그 만들기

외부 데이터는 수집했다는 사실보다 출처, 저장 위치, 수집 시점, 활용 주의사항을 기록하는 것이 중요합니다.


In [ ]:
external_data_log = pd.DataFrame({
    'data_name': [
        'public_data_sample',
        'naver_blog_search_jeju',
        'scraped_example_links',
    ],
    'source': [
        '공공데이터 파일 또는 API',
        '네이버 블로그 검색 API',
        '공개 웹페이지 예시',
    ],
    'save_path': [
        'data/external/public_data_sample.csv',
        'data/external/naver_blog_search_jeju.csv',
        'data/external/scraped_example_links.csv',
    ],
    'collection_method': ['파일 다운로드 또는 공공 API', 'API 호출', '크롤링'],
    'usage_note': [
        '분석 목적에 따라 내부 데이터와 날짜 또는 지역 기준으로 연결 가능',
        '키워드 관심도 참고 자료로 활용 가능하나 대표성 해석 주의',
        '크롤링 구조 이해용 예시이며 실제 사이트 적용 전 정책 확인 필요',
    ],
    'collected_at': ['', '', ''],
})

external_data_log.to_csv(REPORT_DIR / 'ch13_external_data_log.csv', index=False, encoding='utf-8-sig')
external_data_log


## 12. 외부 데이터 수집 요약 보고서 저장

수집 목적, 수집 방법, 내부 데이터와의 연결 기준, 체크리스트, 수집 로그를 Markdown 보고서로 저장합니다.


In [ ]:
summary_text = f'''# Chapter 13 외부 데이터 수집 요약

## 1. 수집 목적

내부 온라인 쇼핑몰 데이터만으로 답하기 어려운 분석 질문을 확장하기 위해 공공데이터, 검색 API, 기본 크롤링 방식의 외부 데이터 수집 흐름을 검토했습니다.

## 2. 외부 데이터 후보

```text
{external_data_plan.to_string(index=False)}
```

## 3. 수집 방법 비교

```text
{collection_method_summary.to_string(index=False)}
```

## 4. 내부 데이터와 연결 기준

```text
{external_integration_plan.to_string(index=False)}
```

## 5. 수집 결과 로그 템플릿

```text
{external_data_log.to_string(index=False)}
```

## 6. 외부 데이터 수집 체크리스트

```text
{external_data_checklist.to_string(index=False)}
```

## 7. 활용 시 주의사항

- 외부 데이터는 출처와 수집 시점을 함께 기록해야 합니다.
- API Key는 .env 파일에 저장하고 GitHub에 올리지 않아야 합니다.
- API 주소, 파라미터, 응답 구조는 공식 문서 기준으로 확인해야 합니다.
- 크롤링은 사이트 정책을 확인한 뒤 제한적으로 사용해야 합니다.
- 외부 데이터와 내부 데이터를 연결할 때 날짜, 지역, 키워드 기준을 맞춰야 합니다.
- 검색 결과나 크롤링 결과를 전체 여론이나 실제 수요로 단정하지 않아야 합니다.

## 8. 다음 단계

수집한 외부 데이터를 기존 EDA, 시각화, 머신러닝 분석에 어떻게 결합할 수 있는지 검토합니다.
'''

summary_path = REPORT_DIR / 'ch13_external_data_summary.md'
summary_path.write_text(summary_text, encoding='utf-8')
print('외부 데이터 수집 요약 보고서 저장 완료:', summary_path)


## 13. 소스 모듈로 전체 준비 자료 생성하기

위에서 단계별로 만든 외부 데이터 수집 준비 자료는 `src/external_data_collection.py`에 함수로 정리되어 있습니다. 전체 파이프라인을 한 번에 실행할 수 있습니다.


In [ ]:
from src.external_data_collection import run_external_data_collection_setup

external_result = run_external_data_collection_setup(
    base_dir=PROJECT_ROOT,
    report_dir=REPORT_DIR,
)

external_result['outputs']['data_plan']


## 14. 스크립트로 한 번에 실행하기

터미널에서 프로젝트 루트 기준으로 아래 명령을 실행하면 13장 외부 데이터 수집 준비 자료가 자동으로 생성됩니다.

```bash
python scripts/run_external_data_collection.py
```


## 15. LLM에게 외부 데이터 수집 코드를 요청하는 프롬프트

LLM은 API 호출 코드 초안을 만들 수 있지만, API 주소, 파라미터, 인증 방식, 응답 구조는 반드시 공식 문서와 비교해야 합니다.

```text
공공데이터 API를 사용해 데이터를 수집하려고 합니다.

API 문서에서 확인한 정보:
- 요청 URL: 여기에 실제 요청 URL 입력
- 인증 방식: serviceKey 파라미터 사용
- 주요 파라미터: pageNo, numOfRows, type, keyword
- 응답 형식: JSON

요청:
1. Python requests를 사용한 API 호출 코드 예시를 작성해 주세요.
2. 응답 상태 코드 확인과 오류 처리를 포함해 주세요.
3. JSON 응답을 pandas DataFrame으로 변환하는 코드를 작성해 주세요.
4. 결과를 data/external 폴더에 CSV로 저장해 주세요.

주의:
- API Key를 코드에 직접 쓰지 말고 .env에서 읽어오게 해 주세요.
- 실제 문서에 없는 파라미터를 만들지 마세요.
- 요청 실패 시 확인할 항목을 함께 설명해 주세요.
```


## 16. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 공공데이터포털에서 CSV 파일 하나를 내려받아 `data/external/`에 저장하고 구조를 확인하세요.
2. 외부 데이터의 출처, 제공 기관, 수집 시점, 업데이트 주기를 기록하세요.
3. 네이버 검색 API 인증 정보를 `.env`에 저장하고 로드 여부만 확인하세요.
4. 관심 키워드 하나를 정해 검색 결과를 수집하고 HTML 태그를 제거하세요.
5. `https://example.com`의 제목과 링크 목록을 추출해 보세요.
6. 수집한 외부 데이터가 내부 쇼핑몰 데이터와 날짜, 지역, 키워드 중 어떤 기준으로 연결될 수 있는지 정리하세요.
7. `ch13_external_data_checklist.csv`의 status와 memo를 직접 채워 보세요.


In [ ]:
# 과제 1. data/external/에 저장한 CSV 파일을 읽고 구조를 확인해 보세요.
# example_path = EXTERNAL_DIR / 'your_external_data.csv'
# external_df = pd.read_csv(example_path)
# external_df.head()


## 17. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 외부 데이터가 필요한 분석 질문 정리
- 파일 다운로드, API 호출, 크롤링 방식 비교
- `data/external/` 폴더 구조 만들기
- API Key를 `.env`로 안전하게 관리하는 방법
- 공공데이터 CSV 읽기 흐름
- 네이버 블로그 검색 API 호출 구조
- HTML 태그 제거
- 기본 크롤링 구조와 주의사항
- 외부 데이터와 내부 데이터 연결 기준 정리
- 외부 데이터 수집 체크리스트와 로그 작성
- `src/external_data_collection.py`와 `scripts/run_external_data_collection.py`로 반복 실행 가능한 구조 만들기

다음 장에서는 이렇게 수집·분석·보고한 결과를 반복 업무 흐름과 파이프라인으로 연결합니다.
